# Tema 3 - Fuentes Reales y Circuitos Equivalentes

**Teoría de Circuitos - ETSI, Universidad de Sevilla**

---

## Objetivos de aprendizaje

- Modelar fuentes reales de tensión y corriente con su resistencia interna
- Trazar la característica U-I de una fuente real y obtener circuito abierto / cortocircuito
- Convertir entre equivalentes Thévenin y Norton
- Calcular el circuito equivalente de Thévenin y Norton de una red arbitraria
- Aplicar el teorema de máxima transferencia de potencia
- Identificar fuentes dependientes y emplear el método de fuente de prueba

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
import schemdraw
import schemdraw.elements as elm

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'   # azul - curvas principales
COLOR_RECTA = '#cb181d'       # rojo - rectas de carga, límites
COLOR_PUNTO = '#238b45'       # verde - puntos de operación
COLOR_SEC = '#a6cee3'         # azul claro - curvas secundarias
COLOR_AUX = '#b2df8a'         # verde claro - auxiliar

print('Configuración lista.')

Configuración lista.


---

## 1. Fuente real de tensión

Una fuente de tensión ideal entrega siempre $U_g$ sin importar la corriente. En la práctica, la fuente tiene una **resistencia interna** $R_v$ en serie que provoca una caída con la corriente:

$$\boxed{U = U_g - R_v \cdot I}$$

| Condición | Resultado |
|-----------|----------|
| **Circuito abierto** ($I=0$) | $U_{ca} = U_g$ |
| **Cortocircuito** ($U=0$) | $I_{cc} = \dfrac{U_g}{R_v}$ |

La **característica U-I** es una recta de pendiente negativa $-R_v$ que parte de $(0, U_g)$ en el eje de tensiones y corta el eje de corrientes en $I_{cc}$.

In [2]:
# Característica U-I de una fuente real de tensión
Ug = 12.0   # V
Rv = 2.0     # Ohm
Icc = Ug / Rv

I = np.linspace(0, Icc, 200)
U = Ug - Rv * I

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(I, U, color=COLOR_PRINCIPAL, lw=2.5, label=r'$U = U_g - R_v \cdot I$')
ax.plot(0, Ug, 'o', color=COLOR_PUNTO, ms=12, zorder=5,
        label=r'Circuito abierto: $U_{ca}=U_g$')
ax.plot(Icc, 0, 's', color=COLOR_RECTA, ms=12, zorder=5,
        label=r'Cortocircuito: $I_{cc}=U_g/R_v$')

ax.annotate(f'$(0,\\; {Ug:.0f}$ V$)$',
            xy=(0, Ug), xytext=(0.8, Ug - 0.5),
            fontsize=12, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))
ax.annotate(f'$({Icc:.0f}$ A$,\\; 0)$',
            xy=(Icc, 0), xytext=(Icc - 1.5, 2),
            fontsize=12, color=COLOR_RECTA, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=2))

ax.fill_between(I, U, alpha=0.08, color=COLOR_PRINCIPAL)
ax.set_xlabel(r'$I$ (A)')
ax.set_ylabel(r'$U$ (V)')
ax.set_title(r'Característica U-I de una fuente real de tensión ($U_g=12$ V, $R_v=2\\;\Omega$)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.3, Icc + 0.5)
ax.set_ylim(-0.5, Ug + 1)
plt.tight_layout()
plt.show()

ValueError: 
Característica U-I de una fuente real de tensión ($U_g=12$ V, $R_v=2\\;\Omega$)
                                                              ^
ParseException: Expected end of text, found '$'  (at char 62), (line:1, col:63)

Error in callback <function _draw_all_if_interactive at 0x71b7d69ab1a0> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
Característica U-I de una fuente real de tensión ($U_g=12$ V, $R_v=2\\;\Omega$)
                                                              ^
ParseException: Expected end of text, found '$'  (at char 62), (line:1, col:63)

ValueError: 
Característica U-I de una fuente real de tensión ($U_g=12$ V, $R_v=2\\;\Omega$)
                                                              ^
ParseException: Expected end of text, found '$'  (at char 62), (line:1, col:63)

<Figure size 1000x600 with 1 Axes>

In [ ]:
# Diagrama: fuente real de tensión (Thévenin)
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_title('Fuente real de tensión (modelo Thévenin)', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += (Vs := elm.SourceV().up().label(r'$U_g$', loc='left'))
d += elm.Resistor().right().label(r'$R_v$')
d += elm.Line().right().length(1)
d += elm.Dot(open=True).label(r'$+$', loc='right')
top_right = d.here
d += elm.Line().down().at(Vs.start).length(0.5)
d += elm.Line().right().tox(top_right)
d += elm.Dot(open=True).label(r'$-$', loc='right')
d.draw()
plt.tight_layout()
plt.show()

---

## 2. Fuente real de corriente

La fuente ideal de corriente entrega siempre $I_g$ independientemente de la tensión en bornes. En la práctica, parte de esa corriente se desvía por una **resistencia interna** $R_p$ en paralelo:

$$\boxed{I = I_g - G_p \cdot U = I_g - \frac{U}{R_p}}$$

| Condición | Resultado |
|-----------|----------|
| **Cortocircuito** ($U=0$) | $I_{cc} = I_g$ |
| **Circuito abierto** ($I=0$) | $U_{ca} = I_g \cdot R_p$ |

La **característica I-U** es una recta de pendiente negativa $-G_p = -1/R_p$.

In [ ]:
# Característica I-U de una fuente real de corriente
Ig = 6.0     # A
Rp = 2.0     # Ohm
Uca = Ig * Rp

U = np.linspace(0, Uca, 200)
I = Ig - U / Rp

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(U, I, color=COLOR_PRINCIPAL, lw=2.5, label=r'$I = I_g - U/R_p$')
ax.plot(0, Ig, 'o', color=COLOR_PUNTO, ms=12, zorder=5,
        label=r'Cortocircuito: $I_{cc}=I_g$')
ax.plot(Uca, 0, 's', color=COLOR_RECTA, ms=12, zorder=5,
        label=r'Circuito abierto: $U_{ca}=I_g \cdot R_p$')

ax.annotate(f'$(0,\\; {Ig:.0f}$ A$)$',
            xy=(0, Ig), xytext=(1.5, Ig - 0.3),
            fontsize=12, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))
ax.annotate(f'$({Uca:.0f}$ V$,\\; 0)$',
            xy=(Uca, 0), xytext=(Uca - 3, 1.5),
            fontsize=12, color=COLOR_RECTA, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=2))

ax.fill_between(U, I, alpha=0.08, color=COLOR_PRINCIPAL)
ax.set_xlabel(r'$U$ (V)')
ax.set_ylabel(r'$I$ (A)')
ax.set_title(r'Característica I-U de una fuente real de corriente ($I_g=6$ A, $R_p=2\\;\Omega$)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, Uca + 1)
ax.set_ylim(-0.3, Ig + 0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Diagrama: fuente real de corriente (Norton)
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_title('Fuente real de corriente (modelo Norton)', fontsize=14, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += (Is := elm.SourceI().up().label(r'$I_g$', loc='left'))
d += elm.Line().right().length(1.5)
mid = d.here
d += elm.Resistor().down().at(mid).label(r'$R_p$', loc='right')
d += elm.Line().left().tox(Is.start)
d += elm.Line().right().at(mid).length(1.5)
d += elm.Dot(open=True).label(r'$+$', loc='right')
top_right = d.here
d += elm.Line().down().at(Is.start).length(0.5)
d += elm.Line().right().length(3)
bot_mid = d.here
d += elm.Line().right().tox(top_right)
d += elm.Dot(open=True).label(r'$-$', loc='right')
d.draw()
plt.tight_layout()
plt.show()

---

## 3. Equivalencia Thévenin ↔ Norton

Dos circuitos son **equivalentes** si producen la **misma característica U-I** en sus bornes. La fuente real de tensión (Thévenin) y la fuente real de corriente (Norton) son equivalentes cuando:

$$\boxed{U_g = R_g \cdot I_g \qquad R_g = R_v = R_p}$$

**Conversión Thévenin → Norton:**
- $I_g = \dfrac{U_g}{R_g}$, $R_p = R_g$

**Conversión Norton → Thévenin:**
- $U_g = R_g \cdot I_g$, $R_v = R_g$

Ambos modelos comparten:
- La misma tensión de circuito abierto: $U_{ca} = U_g = I_g \cdot R_p$
- La misma corriente de cortocircuito: $I_{cc} = U_g / R_v = I_g$
- La misma pendiente en la curva U-I: $-R_g$

In [ ]:
# Comparación Thévenin ↔ Norton
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Thévenin
axes[0].set_title('Equivalente Thévenin', fontsize=14, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += (Vs := elm.SourceV().up().label(r'$U_g$', loc='left'))
d1 += elm.Resistor().right().label(r'$R_g$')
d1 += elm.Line().right().length(1)
d1 += elm.Dot(open=True).label(r'$+$', loc='right')
tr = d1.here
d1 += elm.Line().down().at(Vs.start).length(0.5)
d1 += elm.Line().right().tox(tr)
d1 += elm.Dot(open=True).label(r'$-$', loc='right')
d1.draw()

# Norton
axes[1].set_title('Equivalente Norton', fontsize=14, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += (Is := elm.SourceI().up().label(r'$I_g$', loc='left'))
d2 += elm.Line().right().length(1.5)
mid2 = d2.here
d2 += elm.Resistor().down().at(mid2).label(r'$R_g$', loc='right')
d2 += elm.Line().left().tox(Is.start)
d2 += elm.Line().right().at(mid2).length(1.5)
d2 += elm.Dot(open=True).label(r'$+$', loc='right')
tr2 = d2.here
d2 += elm.Line().down().at(Is.start).length(0.5)
d2 += elm.Line().right().tox(tr2)
d2 += elm.Dot(open=True).label(r'$-$', loc='right')
d2.draw()

plt.tight_layout()
plt.show()

---

## 4. Asociación de fuentes reales

### 4.1 Asociación en serie (modelo Thévenin)

Al conectar $n$ fuentes reales de tensión en **serie**:

$$\boxed{U_{g,eq} = \sum_{k=1}^{n} U_{g,k} \qquad R_{eq} = \sum_{k=1}^{n} R_{v,k}}$$

Las tensiones se suman algebraicamente (pueden restarse si están en sentido opuesto) y las resistencias internas se suman.

### 4.2 Asociación en paralelo (modelo Norton)

Al conectar $n$ fuentes reales de corriente en **paralelo**:

$$\boxed{I_{g,eq} = \sum_{k=1}^{n} I_{g,k} \qquad G_{eq} = \sum_{k=1}^{n} G_{p,k} = \sum_{k=1}^{n} \frac{1}{R_{p,k}}}$$

Las corrientes se suman algebraicamente y las conductancias se suman (equivalente a resistencias en paralelo).

**Truco para el examen:** Para asociar fuentes mixtas, convierte todas al mismo tipo (todas a Thévenin para serie, todas a Norton para paralelo) antes de sumar.

---

## 5. Equivalente de Thévenin

Cualquier red lineal con dos terminales puede sustituirse por una fuente real de tensión equivalente.

### Procedimiento

**Paso 1:** Identificar los terminales A-B donde se quiere el equivalente.

**Paso 2:** Calcular $V_{Th}$ = **tensión de circuito abierto** entre A y B (desconectar la carga).

**Paso 3:** Calcular $R_{eq}$ = resistencia equivalente vista desde A-B con **todas las fuentes independientes desactivadas**:
- Fuentes de tensión $\to$ **cortocircuito** (cable)
- Fuentes de corriente $\to$ **circuito abierto** (eliminar)

$$\boxed{V_{Th} = V_{AB,\text{circuito abierto}} \qquad R_{eq} = R_{AB,\text{fuentes off}}}$$

**Error frecuente:** No se apagan las fuentes dependientes. Las fuentes dependientes **nunca** se apagan porque dependen de una variable del circuito.

In [ ]:
# Procedimiento visual: obtención del equivalente de Thévenin
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Paso 1: circuito original con carga
axes[0].set_title('Paso 1: Red con carga', fontsize=13, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += elm.RBox().right().label('Red\nLineal')
d1 += elm.Line().right().length(1)
d1 += elm.Dot().label('A', loc='top')
tA = d1.here
d1 += elm.Resistor().down().label(r'$R_L$', loc='right')
d1 += elm.Dot().label('B', loc='bottom')
bB = d1.here
d1 += elm.Line().left().tox(d1.elements[0].start)
d1 += elm.Line().up().toy(d1.elements[0].start)
d1.draw()

# Paso 2: circuito abierto para Vth
axes[1].set_title(r'Paso 2: $V_{Th}$ (circuito abierto)', fontsize=13, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += elm.RBox().right().label('Red\nLineal')
d2 += elm.Line().right().length(1)
d2 += elm.Dot(open=True).label('A (+)', loc='top')
tA2 = d2.here
d2 += elm.Gap().down().label(r'$V_{Th}$', loc='right')
d2 += elm.Dot(open=True).label('B (−)', loc='bottom')
bB2 = d2.here
d2 += elm.Line().left().tox(d2.elements[0].start)
d2 += elm.Line().up().toy(d2.elements[0].start)
d2.draw()

# Paso 3: Req con fuentes off
axes[2].set_title(r'Paso 3: $R_{eq}$ (fuentes off)', fontsize=13, fontweight='bold')
d3 = schemdraw.Drawing(canvas=axes[2])
d3 += elm.RBox().right().label('Red\nPasiva')
d3 += elm.Line().right().length(1)
d3 += elm.Dot(open=True).label('A', loc='top')
tA3 = d3.here
d3 += elm.Gap().down().label(r'$R_{eq}$', loc='right')
d3 += elm.Dot(open=True).label('B', loc='bottom')
bB3 = d3.here
d3 += elm.Line().left().tox(d3.elements[0].start)
d3 += elm.Line().up().toy(d3.elements[0].start)
d3.draw()

plt.tight_layout()
plt.show()

---

## 6. Equivalente de Norton

El equivalente de Norton es una fuente de corriente $I_N$ en paralelo con $R_{eq}$.

**Paso 1:** Calcular $I_N$ = **corriente de cortocircuito** entre A y B.

**Paso 2:** $R_{eq}$ se calcula igual que en Thévenin.

$$\boxed{I_N = I_{AB,\text{cortocircuito}} \qquad R_{eq} = R_{AB,\text{fuentes off}}}$$

### Relación Thévenin-Norton

$$\boxed{V_{Th} = R_{eq} \cdot I_N}$$

Esto permite calcular el Norton a partir del Thévenin o viceversa sin necesidad de resolver de nuevo el circuito.

**Truco para el examen:** Si el circuito abierto es difícil de calcular pero el cortocircuito es simple (o viceversa), calcula uno y obtén el otro por la relación $V_{Th} = R_{eq} \cdot I_N$.

---

## 7. Máxima transferencia de potencia

Dado un circuito con equivalente de Thévenin $V_{Th}$, $R_{eq}$ conectado a una carga $R_L$, la potencia disipada en la carga es:

$$P_L = R_L \cdot I^2 = R_L \cdot \left(\frac{V_{Th}}{R_{eq} + R_L}\right)^2$$

Derivando e igualando a cero:

$$\frac{dP_L}{dR_L} = 0 \implies \boxed{R_L = R_{eq}}$$

La **potencia máxima** transferida a la carga es:

$$\boxed{P_{L,\max} = \frac{V_{Th}^2}{4 R_{eq}}}$$

**Nota:** Cuando $R_L = R_{eq}$, el **rendimiento** es del 50 %. La mitad de la potencia se disipa en $R_{eq}$ y la otra mitad en $R_L$.

In [ ]:
# Potencia en la carga vs resistencia de carga
Vth = 12.0   # V
Req = 4.0     # Ohm

RL = np.linspace(0.1, 30, 500)
PL = RL * (Vth / (Req + RL))**2
PL_max = Vth**2 / (4 * Req)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(RL, PL, color=COLOR_PRINCIPAL, lw=2.5, label=r'$P_L = R_L \left(\frac{V_{Th}}{R_{eq}+R_L}\right)^2$')
ax.plot(Req, PL_max, 'o', color=COLOR_PUNTO, ms=14, zorder=5,
        label=r'Máximo: $R_L = R_{eq}$')
ax.axvline(x=Req, color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.7)
ax.axhline(y=PL_max, color=COLOR_RECTA, ls='--', lw=1.5, alpha=0.7)

ax.annotate(f'$P_{{L,max}} = {PL_max:.1f}$ W\n$R_L = R_{{eq}} = {Req:.0f}\\;\\Omega$',
            xy=(Req, PL_max), xytext=(Req + 5, PL_max - 0.5),
            fontsize=12, color=COLOR_PUNTO, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2),
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLOR_PUNTO))

ax.set_xlabel(r'$R_L$ ($\Omega$)')
ax.set_ylabel(r'$P_L$ (W)')
ax.set_title(r'Máxima transferencia de potencia ($V_{Th}=12$ V, $R_{eq}=4\\;\\Omega$)')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 30)
ax.set_ylim(0, PL_max + 1)
plt.tight_layout()
plt.show()

---

#### Ejercicio resuelto: Máxima transferencia de potencia

**Datos:** $V_{Th} = 48$ V, $R_{eq} = 12\;\Omega$

**Paso 1:** Para máxima potencia, $R_L = R_{eq} = 12\;\Omega$.

**Paso 2:** Corriente en el circuito:

$$I = \frac{V_{Th}}{R_{eq} + R_L} = \frac{48}{12 + 12} = 2\;\text{A}$$

**Paso 3:** Potencia máxima:

$$\boxed{P_{L,\max} = \frac{V_{Th}^2}{4 R_{eq}} = \frac{48^2}{4 \times 12} = \frac{2304}{48} = 48\;\text{W}}$$

**Verificación:** $P_L = R_L \cdot I^2 = 12 \times 2^2 = 48$ W $\checkmark$

**Rendimiento:** $\eta = \dfrac{P_L}{P_{total}} = \dfrac{48}{96} = 50\%$

---

## 8. Fuentes dependientes

Las fuentes **dependientes** (o controladas) son fuentes cuyo valor depende de una variable del circuito. Existen cuatro tipos:

| Tipo | Símbolo | Relación | Unidad del parámetro |
|------|---------|----------|---------------------|
| VCVS (tensión controlada por tensión) | Rombo con $+/-$ | $V = \mu \cdot V_x$ | $\mu$ (adimensional) |
| VCCS (corriente controlada por tensión) | Rombo con flecha | $I = g \cdot V_x$ | $g$ (S, siemens) |
| CCVS (tensión controlada por corriente) | Rombo con $+/-$ | $V = r \cdot I_x$ | $r$ ($\Omega$) |
| CCCS (corriente controlada por corriente) | Rombo con flecha | $I = \alpha \cdot I_x$ | $\alpha$ (adimensional) |

### Thévenin/Norton con fuentes dependientes

Las fuentes dependientes **NO se apagan** al calcular $R_{eq}$. En su lugar se usa el **método de la fuente de prueba**:

1. Apagar todas las fuentes **independientes**
2. Conectar una fuente de prueba $V_p$ (o $I_p$) entre A y B
3. Resolver el circuito para obtener $I_p$ (o $V_p$)
4. $R_{eq} = V_p / I_p$

$$\boxed{R_{eq} = \frac{V_p}{I_p}}$$

**Error frecuente:** Apagar una fuente dependiente como si fuera independiente. Esto da un resultado incorrecto.

---

## 9. Ejercicios resueltos del boletín

### 9.1 Problema 1: Thévenin simple

#### Ejercicio resuelto: Equivalente Thévenin (P1)

**Datos:** Red con fuentes independientes. Resultado esperado: $E_{Th} = 100$ V, $R_{eq} = 25\;\Omega$.

**Paso 1:** Calcular $V_{Th}$ (tensión de circuito abierto entre A-B).

Se desconecta la carga y se resuelve el circuito. Aplicando divisor de tensión o mallas:

$$\boxed{V_{Th} = E_{Th} = 100\;\text{V}}$$

**Paso 2:** Calcular $R_{eq}$ apagando fuentes independientes:
- Fuentes de tensión → cortocircuito
- Fuentes de corriente → circuito abierto

$$\boxed{R_{eq} = 25\;\Omega}$$

**Paso 3:** Circuito equivalente: fuente de $100$ V en serie con $25\;\Omega$.

In [ ]:
# Diagrama: equivalente Thévenin P1
fig, ax = plt.subplots(figsize=(7, 5))
ax.set_title(r'Equivalente Thévenin P1: $V_{Th}=100$ V, $R_{eq}=25\;\Omega$',
             fontsize=13, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)
d += (Vs := elm.SourceV().up().label(r'$100$ V', loc='left'))
d += elm.Resistor().right().label(r'$25\;\Omega$')
d += elm.Line().right().length(1)
d += elm.Dot(open=True).label('A', loc='right')
top_r = d.here
d += elm.Line().down().at(Vs.start).length(0.5)
d += elm.Line().right().tox(top_r)
d += elm.Dot(open=True).label('B', loc='right')
d.draw()
plt.tight_layout()
plt.show()

### 9.2 Problema 2: Potencia y rendimiento

#### Ejercicio resuelto: Potencia y rendimiento (P2)

**Datos:** Equivalente Thévenin con $V_{Th} = 100$ V, $R_{eq} = 25\;\Omega$, carga $R_L = 75\;\Omega$.

**Paso 1:** Corriente:

$$I = \frac{V_{Th}}{R_{eq} + R_L} = \frac{100}{25 + 75} = 1\;\text{A}$$

**Paso 2:** Potencia en la carga:

$$P_L = R_L \cdot I^2 = 75 \times 1^2 = 75\;\text{W}$$

**Paso 3:** Potencia total y rendimiento:

$$P_{total} = V_{Th} \cdot I = 100 \times 1 = 100\;\text{W}$$

$$\boxed{\eta = \frac{P_L}{P_{total}} = \frac{75}{100} = 75\%}$$

**Nota:** El rendimiento del 75 % es mayor que el 50 % de máxima transferencia porque $R_L > R_{eq}$. Mayor rendimiento implica menor potencia absoluta en la carga respecto al máximo posible.

### 9.3 Problema 4: Relación $R = 3R_g$ para $P/P_{\max}$

#### Ejercicio resuelto: Fracción de potencia máxima (P4)

**Datos:** $R_L = 3 R_{eq}$. Calcular $P_L / P_{L,\max}$.

**Paso 1:** Potencia con $R_L = 3 R_{eq}$:

$$P_L = 3 R_{eq} \cdot \left(\frac{V_{Th}}{R_{eq} + 3 R_{eq}}\right)^2 = 3 R_{eq} \cdot \frac{V_{Th}^2}{16 R_{eq}^2} = \frac{3 V_{Th}^2}{16 R_{eq}}$$

**Paso 2:** Potencia máxima ($R_L = R_{eq}$):

$$P_{L,\max} = \frac{V_{Th}^2}{4 R_{eq}}$$

**Paso 3:** Razón:

$$\boxed{\frac{P_L}{P_{L,\max}} = \frac{3/16}{1/4} = \frac{3}{4} = 75\%}$$

Con $R_L = 3 R_g$ se obtiene el 75 % de la potencia máxima.

In [ ]:
# Fracción de potencia máxima vs RL/Req
r_ratio = np.linspace(0.05, 10, 500)
P_ratio = 4 * r_ratio / (1 + r_ratio)**2

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(r_ratio, P_ratio, color=COLOR_PRINCIPAL, lw=2.5, label=r'$P_L / P_{L,max}$')
ax.plot(1, 1, 'o', color=COLOR_PUNTO, ms=14, zorder=5, label=r'$R_L = R_{eq}$: 100%')
ax.plot(3, 0.75, 's', color=COLOR_RECTA, ms=12, zorder=5, label=r'$R_L = 3R_{eq}$: 75%')

ax.axhline(y=0.75, color=COLOR_RECTA, ls='--', lw=1, alpha=0.5)
ax.axvline(x=3, color=COLOR_RECTA, ls='--', lw=1, alpha=0.5)

ax.annotate(r'$P_L/P_{max} = 75\%$',
            xy=(3, 0.75), xytext=(5, 0.85),
            fontsize=12, color=COLOR_RECTA, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=COLOR_RECTA, lw=2))

ax.set_xlabel(r'$R_L / R_{eq}$')
ax.set_ylabel(r'$P_L / P_{L,max}$')
ax.set_title(r'Fracción de potencia máxima transferida')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 10)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

### 9.4 Problema 6: Thévenin con red de resistores

#### Ejercicio resuelto: Thévenin con 6 resistores (P6)

**Datos:** Red con 6 resistores y una fuente de tensión. Calcular el equivalente de Thévenin visto desde los terminales A-B.

**Paso 1:** Desconectar la carga de A-B.

**Paso 2:** Calcular $V_{Th}$ por superposición o mallas/nodos. Aplicar divisores de tensión donde sea posible.

**Paso 3:** Apagar fuentes independientes y calcular $R_{eq}$ combinando resistencias en serie y paralelo sucesivamente.

**Método general para $R_{eq}$:**
1. Cortocircuitar fuentes de tensión
2. Abrir fuentes de corriente
3. Simplificar: buscar resistencias en serie (misma corriente) y en paralelo (misma tensión)
4. Repetir hasta obtener una sola resistencia entre A y B

In [ ]:
# Diagrama conceptual: red de 6 resistores para Thévenin
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_title('P6: Red de resistores para equivalente Thévenin', fontsize=13, fontweight='bold')
d = schemdraw.Drawing(canvas=ax)

# Fuente + R1 en serie
d += (Vs := elm.SourceV().up().label(r'$V_s$', loc='left'))
d += elm.Resistor().right().label(r'$R_1$')
n1 = d.here
d += elm.Resistor().right().label(r'$R_2$')
n2 = d.here
d += elm.Line().right().length(1)
d += elm.Dot(open=True).label('A', loc='right')
top_r = d.here

# Rama vertical desde n1
d += elm.Resistor().down().at(n1).label(r'$R_3$', loc='right')
n3 = d.here

# Rama vertical desde n2
d += elm.Resistor().down().at(n2).label(r'$R_4$', loc='right')
n4 = d.here

# Rama inferior
d += elm.Resistor().right().at(n3).label(r'$R_5$')
d += elm.Line().right().tox(n4)

# Conexión inferior
d += elm.Line().left().at(n3).tox(Vs.start)
d += elm.Line().up().toy(Vs.start)

# Terminal B
d += elm.Resistor().down().at(n4).label(r'$R_6$', loc='right')
d += elm.Line().right().tox(top_r)
d += elm.Dot(open=True).label('B', loc='right')

d.draw()
plt.tight_layout()
plt.show()

### 9.5 Problema 7: Equivalente Norton

#### Ejercicio resuelto: Norton (P7)

**Datos:** Red con fuente de tensión y resistencias. Calcular el equivalente de Norton.

**Paso 1:** Cortocircuitar A-B y calcular $I_N$ (corriente que circula por el cortocircuito).

**Paso 2:** Calcular $R_{eq}$ apagando fuentes independientes (igual que Thévenin).

**Paso 3:** El equivalente Norton es $I_N$ en paralelo con $R_{eq}$.

**Verificación:** $V_{Th} = R_{eq} \cdot I_N$ debe coincidir con la tensión de circuito abierto.

### 9.6 Problema 11: Thévenin con fuente dependiente

#### Ejercicio resuelto: Fuente dependiente (P11)

**Datos:** Circuito con fuente dependiente tipo CCVS ($V = r \cdot I_x$). Calcular $V_{ca}$ e $I_{cc}$.

**Paso 1 — Tensión de circuito abierto ($V_{ca}$):**

Con A-B abierto, $I_{AB} = 0$. Resolver el circuito con la fuente dependiente activa. La fuente dependiente **no se apaga**.

**Paso 2 — Corriente de cortocircuito ($I_{cc}$):**

Con A-B en corto, resolver para la corriente que circula por el cortocircuito. La fuente dependiente sigue activa y su valor cambia porque las variables del circuito cambian.

**Paso 3 — Resistencia equivalente:**

$$\boxed{R_{eq} = \frac{V_{ca}}{I_{cc}}}$$

**Alternativa (método de fuente de prueba):**
1. Apagar fuentes independientes
2. Conectar fuente de prueba $V_p$ entre A-B
3. Resolver para $I_p$ con la fuente dependiente activa
4. $R_{eq} = V_p / I_p$

**Error frecuente:** Si se apaga la fuente dependiente, $R_{eq}$ sale incorrecto.

In [ ]:
# Diagrama conceptual: método fuente de prueba
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método Vca / Icc
axes[0].set_title(r'Método $V_{ca}/I_{cc}$', fontsize=13, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += elm.RBox().right().label('Red con\nfuente dep.')
d1 += elm.Line().right().length(1)
d1 += elm.Dot(open=True).label('A', loc='top')
tA = d1.here
d1 += elm.Gap().down().label(r'$R_{eq}=V_{ca}/I_{cc}$', loc='right')
d1 += elm.Dot(open=True).label('B', loc='bottom')
bB = d1.here
d1 += elm.Line().left().tox(d1.elements[0].start)
d1 += elm.Line().up().toy(d1.elements[0].start)
d1.draw()

# Método fuente de prueba
axes[1].set_title(r'Método fuente de prueba ($V_p / I_p$)', fontsize=13, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += elm.RBox().right().label('Red\n(indep. off)')
d2 += elm.Line().right().length(1)
d2 += elm.Dot().label('A', loc='top')
tA2 = d2.here
d2 += elm.SourceV().down().label(r'$V_p$', loc='right').reverse()
d2 += elm.Dot().label('B', loc='bottom')
bB2 = d2.here
d2 += elm.Line().left().tox(d2.elements[0].start)
d2 += elm.Line().up().toy(d2.elements[0].start)
d2.draw()

plt.tight_layout()
plt.show()

---

## 10. Catálogo completo de ejercicios: todos los patrones

| # | Tipo | Componentes | Ecuación clave | Dificultad |
|---|------|-------------|----------------|------------|
| 1 | Thévenin simple (solo resistores + fuente V) | $V_s$, resistores | $V_{Th}$ = divisor de tensión, $R_{eq}$ = serie/paralelo | ⭐ |
| 2 | Norton simple | $V_s$ o $I_s$, resistores | $I_N = I_{cc}$, $R_{eq}$ ídem | ⭐ |
| 3 | Thévenin con fuente de corriente | $I_s$, resistores | $V_{Th} = I_s \cdot R$, convertir a Thévenin | ⭐⭐ |
| 4 | Conversión Thévenin ↔ Norton | $V_{Th}$, $R_{eq}$ | $I_N = V_{Th}/R_{eq}$, $V_{Th} = R_{eq} \cdot I_N$ | ⭐ |
| 5 | Máxima transferencia de potencia | $V_{Th}$, $R_{eq}$, $R_L$ | $R_L = R_{eq}$, $P_{max} = V_{Th}^2/(4R_{eq})$ | ⭐⭐ |
| 6 | Rendimiento y potencia | $V_{Th}$, $R_{eq}$, $R_L$ | $\eta = R_L/(R_{eq}+R_L)$ | ⭐⭐ |
| 7 | Asociación serie de fuentes | Varias $V_{g,k}$, $R_{v,k}$ | $V_{eq} = \Sigma V_k$, $R_{eq} = \Sigma R_k$ | ⭐ |
| 8 | Asociación paralelo de fuentes | Varias $I_{g,k}$, $R_{p,k}$ | $I_{eq} = \Sigma I_k$, $G_{eq} = \Sigma G_k$ | ⭐⭐ |
| 9 | Thévenin con fuente dependiente | Fuente dep. + independientes | Fuente de prueba: $R_{eq} = V_p/I_p$ | ⭐⭐⭐ |
| 10 | Fracción de potencia máxima | $R_L = n \cdot R_{eq}$ | $P/P_{max} = 4n/(1+n)^2$ | ⭐⭐ |

### 10.1 Tipo 1: Thévenin simple (fuente V + resistores)

El caso más básico: una fuente de tensión con resistores. Se calcula $V_{Th}$ por divisor de tensión y $R_{eq}$ por combinación serie/paralelo.

$$\boxed{V_{Th} = V_s \cdot \frac{R_2}{R_1 + R_2}} \qquad \boxed{R_{eq} = R_1 \| R_2 = \frac{R_1 \cdot R_2}{R_1 + R_2}}$$

**Cómo afectan los componentes:**
- Si **$R_1$ aumenta** $\to$ $V_{Th}$ disminuye $\to$ cae más tensión en $R_1$
- Si **$R_2$ aumenta** $\to$ $V_{Th}$ aumenta $\to$ se acerca a $V_s$
- $R_{eq}$ siempre es menor que el menor de $R_1$, $R_2$

#### Ejercicio resuelto: Divisor de tensión

**Datos:** $V_s = 20$ V, $R_1 = 10\;\Omega$, $R_2 = 40\;\Omega$.

**Paso 1:** Tensión de circuito abierto (divisor):

$$V_{Th} = 20 \cdot \frac{40}{10 + 40} = 20 \cdot 0.8 = 16\;\text{V}$$

**Paso 2:** Resistencia equivalente ($V_s \to$ cortocircuito, $R_1 \| R_2$):

$$R_{eq} = \frac{10 \times 40}{10 + 40} = \frac{400}{50} = 8\;\Omega$$

$$\boxed{V_{Th} = 16\;\text{V}, \quad R_{eq} = 8\;\Omega}$$

In [ ]:
# Diagrama: Thévenin simple (divisor de tensión)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Circuito original
axes[0].set_title('Circuito original', fontsize=13, fontweight='bold')
d1 = schemdraw.Drawing(canvas=axes[0])
d1 += (Vs := elm.SourceV().up().label(r'$20$ V', loc='left'))
d1 += elm.Resistor().right().label(r'$R_1=10\;\Omega$')
n1 = d1.here
d1 += elm.Line().right().length(1)
d1 += elm.Dot(open=True).label('A', loc='right')
tA = d1.here
d1 += elm.Resistor().down().at(n1).label(r'$R_2=40\;\Omega$', loc='right')
d1 += elm.Line().left().tox(Vs.start)
d1 += elm.Line().up().toy(Vs.start)
d1 += elm.Line().down().at(n1).at(d1.elements[-2].end).right().tox(tA)
d1 += elm.Dot(open=True).label('B', loc='right')
d1.draw()

# Equivalente
axes[1].set_title(r'Equivalente Thévenin: $16$ V, $8\;\Omega$', fontsize=13, fontweight='bold')
d2 = schemdraw.Drawing(canvas=axes[1])
d2 += (Vth := elm.SourceV().up().label(r'$16$ V', loc='left'))
d2 += elm.Resistor().right().label(r'$8\;\Omega$')
d2 += elm.Line().right().length(1)
d2 += elm.Dot(open=True).label('A', loc='right')
tr = d2.here
d2 += elm.Line().down().at(Vth.start).length(0.5)
d2 += elm.Line().right().tox(tr)
d2 += elm.Dot(open=True).label('B', loc='right')
d2.draw()

plt.tight_layout()
plt.show()

### 10.2 Tipo 2: Norton simple

Se obtiene el equivalente Norton directamente o convirtiendo desde Thévenin.

$$\boxed{I_N = \frac{V_{Th}}{R_{eq}}} \qquad R_{eq} = \text{(misma que Thévenin)}$$

#### Ejercicio resuelto: Norton a partir de Thévenin

**Datos:** $V_{Th} = 16$ V, $R_{eq} = 8\;\Omega$ (del ejercicio anterior).

$$I_N = \frac{V_{Th}}{R_{eq}} = \frac{16}{8} = 2\;\text{A}$$

$$\boxed{I_N = 2\;\text{A}, \quad R_{eq} = 8\;\Omega}$$

### 10.3 Tipo 3: Thévenin con fuente de corriente

Cuando la red contiene fuentes de corriente, se aplica el mismo procedimiento pero la fuente de corriente se **abre** al calcular $R_{eq}$.

$$\boxed{V_{Th} = I_s \cdot R \quad \text{(si la corriente pasa íntegra por } R \text{)}}$$

#### Ejercicio resuelto: Red con fuente de corriente

**Datos:** $I_s = 3$ A, $R_1 = 6\;\Omega$ (en paralelo con $I_s$), $R_2 = 4\;\Omega$ (en serie con el paralelo).

**Paso 1:** $V_{Th}$ (circuito abierto): Toda la corriente $I_s$ pasa por $R_1$ (A-B abierto, no hay camino por $R_2$):

$$V_{Th} = I_s \cdot R_1 + 0 = 3 \times 6 = 18\;\text{V}$$

**Paso 2:** $R_{eq}$: $I_s$ se abre. Queda $R_1$ en serie con $R_2$:

$$R_{eq} = R_1 + R_2 = 6 + 4 = 10\;\Omega$$

$$\boxed{V_{Th} = 18\;\text{V}, \quad R_{eq} = 10\;\Omega}$$

### 10.4 Tipo 4: Conversión Thévenin ↔ Norton

$$\boxed{\text{Th} \to \text{N}: \quad I_N = \frac{V_{Th}}{R_{eq}}, \quad R_N = R_{eq}}$$

$$\boxed{\text{N} \to \text{Th}: \quad V_{Th} = R_{eq} \cdot I_N, \quad R_{Th} = R_{eq}}$$

#### Ejercicio resuelto: Doble conversión

**Datos:** Norton con $I_N = 5$ A, $R_{eq} = 20\;\Omega$.

**Th:** $V_{Th} = 20 \times 5 = 100$ V, $R_{Th} = 20\;\Omega$.

**Verificación (vuelta a Norton):** $I_N = 100 / 20 = 5$ A $\checkmark$

### 10.5 Tipo 5: Máxima transferencia de potencia

$$\boxed{R_L = R_{eq}} \qquad \boxed{P_{L,\max} = \frac{V_{Th}^2}{4 R_{eq}}}$$

**Cómo afectan los parámetros:**
- Si **$V_{Th}$ aumenta** $\to$ $P_{max}$ aumenta cuadráticamente
- Si **$R_{eq}$ aumenta** $\to$ $P_{max}$ disminuye $\to$ fuente menos "potente"

#### Ejercicio resuelto: Máxima potencia

**Datos:** $V_{Th} = 24$ V, $R_{eq} = 6\;\Omega$.

$$R_L = R_{eq} = 6\;\Omega$$

$$P_{L,\max} = \frac{24^2}{4 \times 6} = \frac{576}{24} = 24\;\text{W}$$

$$\boxed{P_{L,\max} = 24\;\text{W} \text{ con } R_L = 6\;\Omega}$$

### 10.6 Tipo 6: Rendimiento y potencia

$$\boxed{\eta = \frac{P_L}{P_{total}} = \frac{R_L}{R_{eq} + R_L}}$$

**Nota:** Máxima potencia ($R_L = R_{eq}$) da $\eta = 50\%$. Para $\eta$ mayor se necesita $R_L \gg R_{eq}$, pero la potencia absoluta baja.

#### Ejercicio resuelto: Rendimiento del 80 %

**Datos:** $V_{Th} = 50$ V, $R_{eq} = 10\;\Omega$. ¿Qué $R_L$ da $\eta = 80\%$?

$$\eta = \frac{R_L}{R_{eq} + R_L} = 0.8 \implies R_L = 0.8 (10 + R_L) \implies 0.2 R_L = 8$$

$$\boxed{R_L = 40\;\Omega}$$

**Verificación:** $I = 50/(10+40) = 1$ A, $P_L = 40 \times 1 = 40$ W, $P_{total} = 50$ W, $\eta = 40/50 = 80\%$ $\checkmark$

### 10.7 Tipo 7: Asociación serie de fuentes

$$\boxed{V_{eq} = V_{g1} + V_{g2} + \cdots} \qquad \boxed{R_{eq} = R_{v1} + R_{v2} + \cdots}$$

(Con signos: sumar si el sentido de la fuente coincide, restar si es opuesto.)

#### Ejercicio resuelto: Dos pilas en serie

**Datos:** Pila 1: $V_{g1} = 1.5$ V, $R_{v1} = 0.5\;\Omega$. Pila 2: $V_{g2} = 1.5$ V, $R_{v2} = 0.3\;\Omega$. Misma polaridad.

$$V_{eq} = 1.5 + 1.5 = 3\;\text{V}$$

$$R_{eq} = 0.5 + 0.3 = 0.8\;\Omega$$

$$\boxed{V_{eq} = 3\;\text{V}, \quad R_{eq} = 0.8\;\Omega}$$

### 10.8 Tipo 8: Asociación paralelo de fuentes

$$\boxed{I_{eq} = I_{g1} + I_{g2} + \cdots} \qquad \boxed{G_{eq} = G_{p1} + G_{p2} + \cdots}$$

#### Ejercicio resuelto: Dos fuentes de corriente en paralelo

**Datos:** $I_{g1} = 2$ A, $R_{p1} = 12\;\Omega$. $I_{g2} = 3$ A, $R_{p2} = 6\;\Omega$.

$$I_{eq} = 2 + 3 = 5\;\text{A}$$

$$G_{eq} = \frac{1}{12} + \frac{1}{6} = \frac{1}{12} + \frac{2}{12} = \frac{3}{12} = 0.25\;\text{S}$$

$$R_{eq} = \frac{1}{G_{eq}} = \frac{1}{0.25} = 4\;\Omega$$

$$\boxed{I_{eq} = 5\;\text{A}, \quad R_{eq} = 4\;\Omega}$$

**Thévenin equivalente:** $V_{Th} = R_{eq} \cdot I_{eq} = 4 \times 5 = 20$ V

### 10.9 Tipo 9: Thévenin con fuente dependiente

**Método obligatorio:** fuente de prueba (las fuentes dependientes NO se apagan).

$$\boxed{R_{eq} = \frac{V_p}{I_p}} \qquad \text{(con fuentes independientes apagadas)}$$

**Cómo afecta la fuente dependiente:**
- Una fuente dependiente puede hacer que $R_{eq}$ sea **mayor** o **menor** que la combinación de resistores pasivos
- En casos extremos, $R_{eq}$ puede ser incluso **negativa** (circuitos con ganancia)

#### Ejercicio resuelto: CCVS con fuente de prueba

**Datos:** $R = 10\;\Omega$, fuente dependiente $V_d = 5 I_x$ (CCVS, $r = 5\;\Omega$), donde $I_x$ es la corriente por $R$.

**Paso 1:** Apagar fuentes independientes. Conectar $V_p$ entre A-B.

**Paso 2:** Por KVL: $V_p = R \cdot I_p + V_d = R \cdot I_p + r \cdot I_x$.

Si $I_x = I_p$ (en serie): $V_p = (R + r) \cdot I_p = (10 + 5) \cdot I_p = 15 I_p$

$$\boxed{R_{eq} = \frac{V_p}{I_p} = 15\;\Omega}$$

Sin la fuente dependiente sería $R_{eq} = 10\;\Omega$. La CCVS **suma** su resistencia equivalente $r$.

### 10.10 Tipo 10: Fracción de potencia máxima

Cuando $R_L = n \cdot R_{eq}$:

$$\boxed{\frac{P_L}{P_{L,\max}} = \frac{4n}{(1+n)^2}}$$

| $R_L / R_{eq}$ | $P_L / P_{L,\max}$ |
|:-:|:-:|
| 0.5 | 88.9 % |
| 1.0 | 100 % |
| 2.0 | 88.9 % |
| 3.0 | 75.0 % |
| 5.0 | 55.6 % |
| 10.0 | 33.1 % |

**Observación:** La curva es simétrica en escala logarítmica respecto a $R_L = R_{eq}$.

#### Ejercicio resuelto: ¿Qué $R_L$ da el 90 % de $P_{max}$?

$$\frac{4n}{(1+n)^2} = 0.9 \implies 4n = 0.9(1+n)^2 = 0.9 + 1.8n + 0.9n^2$$

$$0.9n^2 - 2.2n + 0.9 = 0 \implies n = \frac{2.2 \pm \sqrt{4.84 - 3.24}}{1.8} = \frac{2.2 \pm 1.265}{1.8}$$

$$\boxed{n_1 \approx 0.52 \quad \text{o} \quad n_2 \approx 1.92}$$

Hay dos valores de $R_L$ (uno menor y otro mayor que $R_{eq}$) que dan el mismo porcentaje.

---

## 11. Resumen y tabla de fórmulas clave

| Fórmula | Uso |
|---------|-----|
| $U = U_g - R_v \cdot I$ | Fuente real de tensión (Thévenin) |
| $I = I_g - U / R_p$ | Fuente real de corriente (Norton) |
| $U_g = R_g \cdot I_g$ | Conversión Thévenin ↔ Norton |
| $V_{Th} = V_{AB,\text{c.a.}}$ | Tensión de circuito abierto |
| $I_N = I_{AB,\text{c.c.}}$ | Corriente de cortocircuito |
| $R_{eq} = R_{AB,\text{fuentes off}}$ | Resistencia equivalente (fuentes independientes) |
| $R_{eq} = V_p / I_p$ | Resistencia equivalente (fuentes dependientes) |
| $V_{Th} = R_{eq} \cdot I_N$ | Relación Thévenin-Norton |
| $R_L = R_{eq} \implies P_{max}$ | Condición de máxima transferencia de potencia |
| $P_{L,max} = V_{Th}^2 / (4 R_{eq})$ | Potencia máxima en la carga |
| $\eta = R_L / (R_{eq} + R_L)$ | Rendimiento |
| $P_L/P_{max} = 4n/(1+n)^2$ | Fracción de potencia máxima ($n = R_L/R_{eq}$) |

### Reglas para desactivar fuentes

| Tipo de fuente | Para calcular $R_{eq}$ |
|----------------|----------------------|
| Fuente de tensión independiente | Cortocircuito (cable) |
| Fuente de corriente independiente | Circuito abierto (eliminar) |
| Fuente dependiente (cualquier tipo) | **NO se apaga** → fuente de prueba |

### Tipos de fuentes dependientes

| Tipo | Control | Salida | Parámetro |
|------|---------|--------|-----------|
| VCVS | Tensión | Tensión | $\mu$ (adimensional) |
| VCCS | Tensión | Corriente | $g$ (S) |
| CCVS | Corriente | Tensión | $r$ ($\Omega$) |
| CCCS | Corriente | Corriente | $\alpha$ (adimensional) |